# ColonySearch — Hyperparameter Tuning Benchmark

Compares **QPSO**, **Grid Search**, and **Random Search** on local node search quality.
All nodes in `data/dbs/` are used together so the optimiser sees every topic cluster.

Metric: **NDCG@10** — ground truth = documents sharing the same topic label on the same node.

In [ ]:
import sys
from pathlib import Path

# Walk up from CWD until we find the project root (works wherever Jupyter is launched)
_ROOT = Path().resolve()
for _ in range(6):
    if (_ROOT / 'data' / 'dbs').exists():
        break
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from data.tuning.ground_truth import load_all_ground_truth
from data.tuning.metrics import evaluate
from data.tuning.algos import BOUNDS
from data.tuning.algos.qpso import QPSO
from data.tuning.algos.grid_search import GridSearch
from data.tuning.algos.random_search import RandomSearch

DATA_DIR = _ROOT / 'data'
K = 10
print(f'Project root: {_ROOT}')

## Parameter Bounds

These are the search ranges for all three algorithms.  
Edit this cell to explore different parts of the space.

In [ ]:
# BOUNDS is imported from algos/__init__.py — one source of truth.
# Format: [(min, max), ...] for [link_bias, svd_dims, alpha]
print('Parameter bounds:')
labels = ['link_bias  (link-graph weight vs text embeddings)', 'svd_dims   (link-graph SVD dimensions, rounded to int)', 'alpha      (BM25 weight in final score; 1-alpha = KNN weight)']
for (lo, hi), label in zip(BOUNDS, labels):
    print(f'  {label}')
    print(f'      range: [{lo}, {hi}]')

# You can override bounds for a specific algo by passing them explicitly:
#   QPSO(bounds=[(0.0, 1.0), (8, 64), (0.0, 1.0)])  <- narrower search
print()
print('Current BOUNDS:', BOUNDS)

In [ ]:
print('Loading ground truth across all nodes ...')
test_cases = load_all_ground_truth(DATA_DIR)
unique_clusters = len({frozenset(tc['relevant_urls']) for tc in test_cases})
print(f'\nTotal: {len(test_cases)} test queries across {unique_clusters} topic clusters')

## Algorithm Configuration

Tune budgets here before running. Rough time estimate per evaluation: ~2–5 s on Node_2 (370 docs).

| Algorithm | Default evals | Approx. time |
|-----------|-------------|---------------|
| QPSO      | particles × iterations = 600 | ~20 min |
| Grid      | n_per_dim³ = 125 | ~10 min |
| Random    | n_evaluations = 200 | ~15 min |

For a **quick test** run, use the commented-out small-budget versions below.

In [ ]:
def make_objective(test_cases, k):
    def objective(params):
        return -evaluate(params, test_cases, k=k)['mean_ndcg']
    return objective

objective_fn = make_objective(test_cases, K)

# ── Full budget (comment out for quick test) ──────────────────────────────
algos = [
    ('QPSO',   QPSO(n_particles=30, max_iter=20)),
    ('Grid',   GridSearch(n_per_dim=5)),
    ('Random', RandomSearch(n_evaluations=200)),
]

# ── Quick test (~50 evals total, a few minutes) ───────────────────────────
# algos = [
#     ('QPSO',   QPSO(n_particles=5, max_iter=4)),
#     ('Grid',   GridSearch(n_per_dim=3)),
#     ('Random', RandomSearch(n_evaluations=27)),
# ]

import time
results = {}
for name, algo in algos:
    t0 = time.time()
    print(f'Running {name} ...')
    results[name] = algo.optimize(objective_fn)
    elapsed = time.time() - t0
    print(f'  Best NDCG@{K}: {-results[name].best_fitness:.4f}  ({results[name].n_evals} evals, {elapsed:.0f}s)')

print('\nDone.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

colors = {'QPSO': 'steelblue', 'Grid': 'gray', 'Random': 'darkorange'}
for name, res in results.items():
    xs = [h['n_evals']   for h in res.history]
    ys = [h['best_ndcg'] for h in res.history]
    ax.plot(xs, ys, label=name, color=colors[name], linewidth=2)

ax.set_xlabel('Number of evaluations')
ax.set_ylabel('Best NDCG@10')
ax.set_title('Algorithm Convergence Comparison')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('benchmark_convergence.png', dpi=120)
plt.show()

In [ ]:
rows = []
for name, res in results.items():
    lb, svd, alpha = res.best_params
    metrics = evaluate(res.best_params, test_cases, k=K)
    rows.append({
        'Algorithm':         name,
        'Best NDCG@10':      round(metrics['mean_ndcg'], 4),
        'Best Precision@10': round(metrics['mean_precision'], 4),
        'Best Recall@10':    round(metrics['mean_recall'], 4),
        'Total Evals':       res.n_evals,
        'Best Params':       f'lb={lb:.2f}, svd={int(round(svd))}, a={alpha:.2f}',
    })

df = pd.DataFrame(rows).set_index('Algorithm')
display(df.style.highlight_max(subset=['Best NDCG@10', 'Best Precision@10', 'Best Recall@10']))

## Grid Search Heatmap

NDCG@10 across `alpha × link_bias` with `svd_dims` fixed at the best value found.

In [ ]:
grid_res = results['Grid']
best_lb, best_svd, best_alpha = grid_res.best_params
best_svd_int = int(round(best_svd))

n = 5
lb_vals    = np.linspace(0.0, 1.0, n)
alpha_vals = np.linspace(0.0, 1.0, n)

print(f'Sweeping alpha x link_bias (svd_dims={best_svd_int}) ...')
ndcg_grid = np.zeros((n, n))
for i, lb in enumerate(lb_vals):
    for j, al in enumerate(alpha_vals):
        m = evaluate([lb, best_svd_int, al], test_cases, k=K)
        ndcg_grid[i, j] = m['mean_ndcg']

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(ndcg_grid, origin='lower', aspect='auto',
               extent=[0, 1, 0, 1], cmap='viridis')
fig.colorbar(im, ax=ax, label='NDCG@10')
ax.set_xlabel('alpha  (0=pure KNN, 1=pure BM25)')
ax.set_ylabel('link_bias  (0=text only, 1=max link influence)')
ax.set_title(f'Grid Search: NDCG@10 by alpha x link_bias  (svd_dims={best_svd_int})')
fig.tight_layout()
plt.show()

## QPSO Particle Trace

Each point is one evaluated position in the `link_bias × alpha` plane (svd_dims collapsed).
Colour = NDCG@10 at that point; red star = global best.

In [ ]:
positions = []

def recording_objective(params):
    m = evaluate(params, test_cases, k=K)
    positions.append((float(params[0]), float(params[2]), m['mean_ndcg']))
    return -m['mean_ndcg']

print('Running QPSO trace (separate run for visualisation) ...')
qpso_trace = QPSO(n_particles=20, max_iter=10, seed=99)
trace_res = qpso_trace.optimize(recording_objective)

lbs, alphas, ndcgs = zip(*positions)
best_lb, _, best_alpha = trace_res.best_params

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(lbs, alphas, c=ndcgs, cmap='plasma', s=20, alpha=0.6)
fig.colorbar(sc, ax=ax, label='NDCG@10')
ax.scatter([best_lb], [best_alpha], marker='*', s=300, color='red', zorder=5, label='gbest')
ax.set_xlabel('link_bias')
ax.set_ylabel('alpha')
ax.set_title('QPSO Search Trajectory')
ax.legend()
fig.tight_layout()
plt.show()
print(f'QPSO trace best: NDCG={-trace_res.best_fitness:.4f}  params={[round(x,3) for x in trace_res.best_params]}')